In [ ]:
import os
import ast
import pandas as pd
from tqdm import tqdm

In [ ]:
val_fold = 9
test_fold = 10

ptbxl_data = "/opt/gpudata/ecg/ptb-xl"
subset_root = "/opt/gpudata/ecg/temp" # path to make subset directories

In [ ]:
df = pd.read_csv(os.path.join(ptbxl_data, "ptbxl_database.csv"), index_col="ecg_id")
original_cols = df.columns

In [ ]:
df["strat_fold"].value_counts()

In [ ]:
def make_train_subset_from_fold(df: pd.DataFrame, train_folds: list[int]):
    # make subsets of training data
    subset = df[df["strat_fold"].isin(train_folds + [val_fold, test_fold])]
    train_size = subset["strat_fold"].value_counts().loc[train_folds].sum()

    train_size_k = train_size // 1024
    if train_size_k == 0:
        train_size_k = str(train_size)
    else:
        train_size_k = f"{train_size_k}k"
    subset_path = os.path.join(subset_root, f"ptb-xl-{train_size_k}")
    os.makedirs(subset_path, exist_ok=True)

    subset.to_csv(os.path.join(subset_path, "ptbxl_database.csv"))

    # also link source waveform data
    for p in [
        "records100",
        "records500",
    ]:
        os.symlink(
            src=os.path.join(ptbxl_data, p),
            dst=os.path.join(subset_path, p),
        )

In [ ]:
for train_folds in tqdm(
    [
        [1, 2, 3, 4],
        [1, 2],
        [1],
    ]
):
    make_train_subset_from_fold(df, train_folds)

### smaller PTB-XL subsets

difficult to use the same stratification algorithm developed by the PTB-XL authors to very small dataset sizes, so we use a coarser stratification strategy after exhausting the author provided splits:
* author provided "diagnostic" labels: NORM, MI, STTC, HYP, CD
* patient sex - same as author
* patient age (bins of 20 years) - same as author
to ensure stratification to the smallest subset (~2^8 samples), we bin samples with unique(-ish) labels together

In [ ]:
import numpy as np
from _ptbxl_stratified_sampling import stratify

In [ ]:
temp = pd.cut(df["age"], [0, 20, 40, 60, 80, 1000]) # right edge to account to age censoring >= 90
age_one_hot = pd.get_dummies(temp, prefix="age").astype(int)
sex_one_hot = pd.get_dummies(df["sex"], prefix="sex")

In [ ]:
# prepare coarse labels for stratification
agg_df = pd.read_csv(os.path.join(ptbxl_data, "scp_statements.csv"), index_col=0)
agg_df = agg_df[agg_df["diagnostic"] == 1]

df["scp_codes_raw"] = df["scp_codes"].copy()
df["scp_codes"] = df["scp_codes"].apply(lambda x: ast.literal_eval(x))

def aggregate_diagnostic(scp_codes: dict):
    tmp = []
    for key in scp_codes.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key, "diagnostic_class"])
    return list(set(tmp))

labels = df["scp_codes"].apply(aggregate_diagnostic)
labels = labels.apply(lambda x: pd.Series(1, index=x)).fillna(0).astype(int)

In [ ]:
# only make smaller subsets of strat 1 (the smallest subset from above) so each subset is nested
split_1_mask = df["strat_fold"] == 1

one_hot_classes = pd.concat([age_one_hot, sex_one_hot, labels], axis=1)
one_hot_classes_by_patient = one_hot_classes.loc[split_1_mask].groupby(df.loc[split_1_mask, "patient_id"]).max()
ecgs_per_patient = df.loc[split_1_mask, "patient_id"].value_counts().sort_index().to_list()

label_lists = [np.where(row)[0].tolist() for row in one_hot_classes_by_patient.to_numpy()]

In [ ]:
n_patients, n_classes = one_hot_classes.shape

# stratify creates a patient ID based on the passed in list
# since we use a subset of the entire dataset, we need a way to map back to the IDs
# of the original whole dataset
remap = {i: v for i, v in enumerate(one_hot_classes_by_patient.index)}

stratified_ids, stratified_labels = stratify(
    data=label_lists,
    classes=list(range(n_classes)),
    ratios=[0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125],
    qualities=[2] * n_patients, # no notion of quality, just fake it with 
    ecgs_per_patient=ecgs_per_patient,
    nr_clean_folds=0,
)

In [ ]:
remapped_ids = []
for subset_ids in stratified_ids:
    subset_ids = [remap[x] for x in subset_ids]
    remapped_ids.append(subset_ids)

In [ ]:
val_test_ids = df.loc[df["strat_fold"].isin({9, 10}), "patient_id"].to_list()

def make_train_subset_small(subfolds: list[int], _name: str):
    # these are patient ids
    train_ids = [x for subfold in subfolds for x in remapped_ids[subfold]]
    subset_df = df.loc[
        df["patient_id"].isin(train_ids + val_test_ids),
        original_cols,
    ]

    subset_path = os.path.join(subset_root, f"ptb-xl-{_name}")
    os.makedirs(subset_path, exist_ok=True)

    subset_df.to_csv(os.path.join(subset_path, "ptbxl_database.csv"))

    # also link source waveform data
    for p in [
        "records100",
        "records500",
    ]:
        os.symlink(
            src=os.path.join(ptbxl_data, p),
            dst=os.path.join(subset_path, p),
        )

In [ ]:
for subfolds, _name in [
    ([0, 1, 2, 3], "1k"),
    ([0, 1], "512"),
    ([0], "256"),
]:
    make_train_subset_small(subfolds, _name)

In [ ]:
pd.read_csv("/opt/gpudata/ecg/temp/ptb-xl-256/ptbxl_database.csv")["strat_fold"].value_counts()